# 01 · Veri Keşfi (Data Exploration)

Amaç: `data/` klasörünü model geliştirmeden önce tanımak — yapı, sınıf dağılımı, görsel boyutları,
bozuk/yinelenen dosyalar ve veri sızıntısı riskleri. Bu notebook veri dosyalarını **değiştirmez**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

## 1. Ortam ve paket kontrolü

In [ ]:
import platform
from importlib.metadata import PackageNotFoundError, version

import torch

print("Python  :", sys.version.split()[0], "|", platform.platform())
for package in ("torch", "numpy", "pillow", "scikit-learn", "matplotlib", "pandas"):
    try:
        print(f"{package:<13}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<13}: NOT INSTALLED")
print("MPS (Apple Silicon) available:", torch.backends.mps.is_available())

## 2. Klasör yapısı ve dosya türleri

In [ ]:
from collections import Counter

import pandas as pd

from passive_radar.config import CLASS_NAMES, ProjectConfig

config = ProjectConfig.default(PROJECT_ROOT)
paths = config.paths

rows = []
for directory in sorted(p for p in paths.data_dir.rglob("*") if p.is_dir()):
    files = [f for f in directory.iterdir() if f.is_file()]
    rows.append({
        "folder": directory.relative_to(paths.data_dir).as_posix(),
        "files": len(files),
        "extensions": dict(Counter(f.suffix.lower() or f.name for f in files)),
    })
pd.DataFrame(rows)

`data/train` iki sınıf klasörü içerir (`target`, `no_target`). `data/test` **etiketsizdir** (alt klasör yok).
`.DS_Store` gibi gizli dosyalar indekslemede yok sayılır.

## 3. Sınıf başına görsel sayısı

In [ ]:
import matplotlib.pyplot as plt

from passive_radar.data.indexing import scan_labeled_directory, scan_unlabeled_directory

labeled = scan_labeled_directory(paths.labeled_dir)
unlabeled = scan_unlabeled_directory(paths.unlabeled_dir)

labeled_df = pd.DataFrame({
    "file": [r.path.name for r in labeled],
    "class": [CLASS_NAMES[r.label] for r in labeled],
    "source_id": [r.source_id for r in labeled],
    "md5": [r.md5 for r in labeled],
})
class_counts = labeled_df["class"].value_counts().reindex(CLASS_NAMES)
print(class_counts.to_string(), f"\nunlabeled (data/test): {len(unlabeled)}")

ax = class_counts.plot.bar(color=["#4C72B0", "#DD8452"], rot=0, title="Images per class (data/train)")
ax.set_ylabel("images")
plt.show()

## 4. Görsel boyutları, kanal modları ve bozuk dosya kontrolü

In [ ]:
from passive_radar.data.preprocessing import inspect_image

all_paths = [r.path for r in labeled] + [r.path for r in unlabeled]
infos = pd.DataFrame([vars(inspect_image(p)) for p in all_paths])
infos["split"] = ["train"] * len(labeled) + ["test (unlabeled)"] * len(unlabeled)

print("Unreadable / corrupt files:", int((~infos["is_valid"]).sum()))
infos.groupby(["split", "width", "height", "mode"]).size().rename("count").reset_index()

Tüm görseller aynı boyutta (2240 × 740, RGBA). Görseller ham radar matrisleri değil, **Plotly ile çizilmiş
figürlerdir**: eksenler, eksen başlıkları, beyaz kenar boşlukları ve bir renk çubuğu (colorbar) içerirler.

## 5. Örnek Range-Doppler görselleri

In [ ]:
import numpy as np

from passive_radar.data.preprocessing import load_rgb_image
from passive_radar.evaluation.plots import plot_image_grid

rng = np.random.default_rng(config.split.seed)
samples = [labeled[i] for c in (0, 1) for i in rng.choice(
    [j for j, r in enumerate(labeled) if r.label == c], size=2, replace=False)]
plot_image_grid(
    [np.asarray(load_rgb_image(r.path)) for r in samples],
    [f"{CLASS_NAMES[r.label]} | {r.path.name[:70]}" for r in samples],
    columns=2,
)
plt.show()

## 6. Yinelenen görseller (byte düzeyinde aynı dosyalar)

In [ ]:
unlabeled_df = pd.DataFrame({"file": [r.path.name for r in unlabeled], "class": "unlabeled",
                             "source_id": [r.source_id for r in unlabeled], "md5": [r.md5 for r in unlabeled]})
everything = pd.concat([labeled_df.assign(split="train"), unlabeled_df.assign(split="test")], ignore_index=True)
dup_groups = everything[everything.duplicated("md5", keep=False)].groupby("md5")

summary = Counter()
for _, group in dup_groups:
    kinds = tuple(sorted(set(group["split"] + "/" + group["class"])))
    summary[kinds] += 1
pd.DataFrame([{"where": " & ".join(k), "duplicate groups": v} for k, v in summary.items()])

In [ ]:
train_test_overlap = set(labeled_df["md5"]) & set(unlabeled_df["md5"])
print(f"data/test images that are byte-identical to a data/train image: {len(train_test_overlap)} / {len(unlabeled)}")
print("Recordings in data/test:", sorted(set(unlabeled_df["source_id"])))
print("...also present in data/train:", sorted(set(unlabeled_df["source_id"]) & set(labeled_df["source_id"])))

## 7. Kaynak kayıtlar (veri sızıntısı riski)

Dosya adları `<kayıt adı> (128kbit_AAC)PartNNN.png` biçimindedir: her görsel bir ses kaydının bir segmentinden
üretilmiştir. Aynı kaydın segmentleri benzer görünür; bu nedenle aynı kaydın train ve test'e dağılması
**veri sızıntısı** yaratır. `source_id`, segment numarası ve sonekler atılarak (Unicode NFC normalizasyonu ile)
türetilir.

In [ ]:
per_source = labeled_df.groupby("source_id")["class"].agg(images="size", classes=lambda s: ",".join(sorted(set(s))))
print("Recordings:", len(per_source))
print("Recordings that contain BOTH classes:", int((per_source["classes"] == "no_target,target").sum()))
print("Images per recording: min", per_source["images"].min(), "| median", int(per_source["images"].median()),
      "| max", per_source["images"].max())
per_source.sort_values("images", ascending=False).head(8)

## 8. Dosya adı / etiket ilişkisi (olası kısayol sinyalleri)

In [ ]:
labeled_df["suffix"] = labeled_df["file"].str.extract(r"Part\d+(.*)\.png$", expand=False).fillna("?")
labeled_df["suffix"] = labeled_df["suffix"].replace("", "(none)")
pd.crosstab(labeled_df["suffix"], labeled_df["class"])

## 9. Bulgular ve kararlar

* **Sınıf dengesi:** `target` ve `no_target` eşit sayıda (510 / 510) → sınıf ağırlığına gerek yok, yine de
  yalnızca accuracy değil precision/recall/F1 raporlanacak.
* **Bozuk dosya yok**, tüm görseller 2240 × 740 RGBA.
* **Yinelenenler:** train içinde birebir kopyalar var (çoğu `(1)`/`(2)` ekli). Dosyalar silinmez, yalnızca
  indeksten çıkarılır (`deduplicate_records`).
* **`data/test` bağımsız değil ve etiketsiz:** tek bir kayıttan (Sam Altman TED2025) geliyor, bu kayıt train'de de
  var ve görsellerin bir kısmı train'dekilerle birebir aynı. Bu yüzden **değerlendirme için `data/train` kayıt
  bazında (grup) train/val/test'e bölünür**; `data/test` yalnızca etiketsiz tahmin demosu olarak kullanılır.
* **Grup sızıntısı:** kayıtların yaklaşık yarısı her iki sınıfı da içeriyor → görsel bazında rastgele bölme yanlış olur.
* **Dosya adı soneki** (`_range_doppler`, `(1)`) sınıfla ilişkili; model dosya adını görmediği için sorun değil
  ama veri üretim sürecinde iki farklı pipeline kullanılmış olabileceğine işaret ediyor.
* **Etiket anlamı (kullanıcıdan teyit edildi):** `target` = ses kaydı aydınlatıcı olarak kullanılan simülasyonda
  segmente yapay hedef yankısı eklenmiş.